In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Data Reading


In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@adbstoragepro.dfs.core.windows.net/products")
display(df)

In [0]:
df = df.drop("_rescued_data")
display(df)

In [0]:
df.createOrReplaceTempView("products")

### Fucntions

In [0]:
 %sql
CREATE OR REPLACE FUNCTION DEV.BRONZE.DISCOUNT_FUNCTION(P_PRICE DOUBLE)
RETURNS DOUBLE
RETURN P_PRICE * 0.09

In [0]:
%sql
Select product_id ,price, DEV.BRONZE.DISCOUNT_FUNCTION(price) from products

In [0]:
df = df.withColumn("discounted_price", expr('DEV.BRONZE.DISCOUNT_FUNCTION(price)'))
display(df)

In [0]:
%sql
CREATE OR REPLACE FUNCTION  DEV.BRONZE.UPPER_FUN(P_BRAND string)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
return p_brand.upper()  
$$

In [0]:
%sql
select product_id, brand, DEV.BRONZE.UPPER_FUN(brand) from products

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@adbstoragepro.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DEV.SILVER.products_silver
    USING DELTA
    LOCATION 'abfss://silver@adbstoragepro.dfs.core.windows.net/products';
